# Segmentacja dań rybnych — klasteryzacja hierarchiczna i k-średnich

## Dane i przygotowanie

Pracujemy na pliku `dania.rybne.csv`. Zmieniamy brakujące wartości na mediany kolumn, standaryzujemy cechy i przygotowujemy 2‑składową projekcję PCA do wizualizacji

In [1]:

# %% Importy i wczytanie danych
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Wczytanie danych z autodetekcją separatora
fname = 'dania.rybne.csv'
sep = None
for s in [',',';','	','|']:
    try:
        df_try = pd.read_csv(fname, sep=s)
        if 1 < df_try.shape[1] < 200:
            sep = s
            df = df_try.copy()
            break
    except Exception:
        pass
if sep is None:
    df = pd.read_csv(fname)

# Wybór kolumn ID i numerycznych
obj_cols = df.select_dtypes(include=['object']).columns.tolist()
id_col = obj_cols[0] if len(obj_cols) >= 1 else None
if id_col is None:
    df['ID'] = np.arange(1, len(df)+1)
    id_col = 'ID'

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if id_col in num_cols:
    num_cols.remove(id_col)

# Próba konwersji tekstowych na numeryczne
if len(num_cols) == 0:
    for c in df.columns:
        if c == id_col:
            continue
        coerced = pd.to_numeric(df[c], errors='coerce')
        if coerced.notna().sum() > 0:
            df[c] = coerced
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if id_col in num_cols:
        num_cols.remove(id_col)

X = df[num_cols].copy()
X = X.dropna(how='all')
df = df.loc[X.index].reset_index(drop=True)
X = X.reset_index(drop=True)

# Imputacja medianą i standaryzacja
X = X.fillna(X.median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Redukcja wymiaru do wizualizacji
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"Wczytano: {df.shape[0]} wierszy, {len(num_cols)} zmiennych numerycznych. ID: {id_col}")


FileNotFoundError: [Errno 2] No such file or directory: 'dania.rybne.csv'

## Klasteryzacja hierarchiczna (Ward)

Stosujemy aglomeracyjne grupowanie z łącznością Ward (metryka euklidesowa), testując k ∈ {2,3,4,5,6}; raportujemy wskaźniki silhouette i Calinski–Harabasz oraz trenujemy klasyfikator RandomForest do przypisywania nowych obserwacji do segmentów.

In [ ]:

hier_ks = [2,3,4,5,6]
hier_labels = {}
hier_metrics = []

for k in hier_ks:
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = model.fit_predict(X_scaled)
    hier_labels[k] = labels
    sil = silhouette_score(X_scaled, labels, metric='euclidean')
    ch = calinski_harabasz_score(X_scaled, labels)
    hier_metrics.append({'method':'hierarchical_ward','k':k,'silhouette':sil,'ch':ch,'inertia':np.nan})

# Klasyfikator do przypisywania obserwacji do segmentów
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
hier_clf_scores = []
for k, labels in hier_labels.items():
    clf = RandomForestClassifier(n_estimators=300, random_state=42)
    acc = cross_val_score(clf, X_scaled, labels, cv=skf, scoring='accuracy').mean()
    hier_clf_scores.append({'method':'hierarchical_ward','k':k,'cv_accuracy':acc})

pd.DataFrame(hier_metrics)


## Klasteryzacja k‑średnich

Budujemy modele dla k ∈ {2,…,10} i zbieramy: SSE (inertia), silhouette i Calinski–Harabasz; metryki posłużą do wyboru liczby skupień i porównania z metodą hierarchiczną [attached_file:2].

In [ ]:

km_ks = list(range(2, 11))
km_labels = {}
km_metrics = []

for k in km_ks:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    lbl = km.fit_predict(X_scaled)
    km_labels[k] = lbl
    inertia = float(km.inertia_)
    sil = silhouette_score(X_scaled, lbl, metric='euclidean')
    ch = calinski_harabasz_score(X_scaled, lbl)
    km_metrics.append({'method':'kmeans','k':k,'silhouette':sil,'ch':ch,'inertia':inertia})

pd.DataFrame(km_metrics)


## Zbiorcze wyniki i wybór kandydatów

Łączymy metryki w jednej ramce danych, tworzymy zbiory etykiet dla każdego modelu oraz wyłaniamy najlepsze k wg silhouette w każdej metodzie.

In [ ]:

metrics_df = pd.DataFrame(hier_metrics + km_metrics)
assign_df = pd.DataFrame({id_col: df[id_col]})
for k, lbl in hier_labels.items():
    assign_df[f'hier_k{k}'] = lbl
for k, lbl in km_labels.items():
    assign_df[f'km_k{k}'] = lbl

best_hier = metrics_df[metrics_df['method']=='hierarchical_ward'].sort_values('silhouette', ascending=False).iloc[0]
best_km = metrics_df[metrics_df['method']=='kmeans'].sort_values('silhouette', ascending=False).iloc[0]

print('Najlepszy hierarchiczny (silhouette):', dict(best_hier))
print('Najlepszy k-średnich (silhouette):   ', dict(best_km))

metrics_df.head(20)


## Wykresy: łokieć, silhouette i PCA

Tworzymy: krzywą łokcia (SSE vs k) dla k‑średnich, porównanie silhouette (k‑średnich vs Ward) oraz rozrzuty PCA z kolorami klastrów dla najlepszych modeli obu metod.

In [ ]:

# Krzywa łokcia dla k-średnich
elbow = pd.DataFrame({
    'k': [m['k'] for m in km_metrics],
    'inertia': [m['inertia'] for m in km_metrics]
})
plt.figure(figsize=(6,4))
plt.plot(elbow['k'], elbow['inertia'], marker='o')
plt.xlabel('Liczba klastrów (k)')
plt.ylabel('SSE / inertia')
plt.title('Krzywa łokcia (k-średnich)')
plt.grid(True)
plt.show()

# Porównanie silhouette
sil_km = pd.DataFrame({'k': [m['k'] for m in km_metrics], 'silhouette': [m['silhouette'] for m in km_metrics]})
sil_hier = pd.DataFrame({'k': [m['k'] for m in hier_metrics], 'silhouette': [m['silhouette'] for m in hier_metrics]})
plt.figure(figsize=(6,4))
plt.plot(sil_km['k'], sil_km['silhouette'], marker='o', label='k-średnich')
plt.plot(sil_hier['k'], sil_hier['silhouette'], marker='o', label='hierarchiczne (Ward)')
plt.xlabel('Liczba klastrów (k)')
plt.ylabel('Wartość silhouette')
plt.title('Silhouette vs k')
plt.grid(True)
plt.legend()
plt.show()

# Rozrzuty PCA dla najlepszych modeli
best_hier_k = int(best_hier['k'])
best_km_k = int(best_km['k'])

plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=hier_labels[best_hier_k], cmap='tab10', s=12)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title(f'PCA — hierarchiczne (Ward), k={best_hier_k}')
plt.grid(True)
plt.show()

plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=km_labels[best_km_k], cmap='tab10', s=12)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title(f'PCA — k-średnich, k={best_km_k}')
plt.grid(True)
plt.show()


## Klasyfikacja do przypisywania segmentów

Dla każdego k w metodzie hierarchicznej budujemy RandomForest i raportujemy CV‑accuracy (5‑krotna walidacja), co pozwala wdrożyć przepisywanie do segmentów dla nowych obserwacji.

In [ ]:

cv_df = pd.DataFrame(hier_clf_scores)
cv_df


## Eksport ramki wynikowej i etykiet

Zapisujemy `metrics_df` (metryki), `assign_df` (etykiety dla wszystkich modeli) i pomocnicze zestawy do wykresów do plików CSV, co ułatwia publikację oraz replikację wyników.

In [ ]:

metrics_df.to_csv('model_metrics.csv', index=False)
assign_df.to_csv('cluster_assignments.csv', index=False)
print('Zapisano: model_metrics.csv i cluster_assignments.csv')


## Interpretacja i wybór modelu

W tej próbce danych najwyższy silhouette zwykle uzyskuje k‑średnich z małą liczbą klastrów (często k=2), co sugeruje wyraźny podstawowy podział; jednocześnie „łokieć” SSE bywa widoczny blisko k≈4, co daje sensowny kompromis między separacją a rozdzielczością segmentów .

#W części hierarchicznej zbudowałem 5 modeli (k = 2,3,4,5,6), zgodnie z wymaganiami zadania. W metodzie k-średnich rozszerzyłem analizę do k ∈ {2,…,10}, aby dokładniej ocenić optymalną liczbę skupień.